In [ ]:
import os
import io
import streamlit as st
from PIL import Image, ImageFilter
from rembg import remove

st.set_page_config(page_title="Smart Background Editor", layout="centered")

st.title("Smart Background Editor")

app_mode = st.selectbox("Select Editing Mode", ["Background Removal", "Background Blur"])

uploaded_file = st.file_uploader("Upload Image to Process", type=["png", "jpg", "jpeg"], key="user_photo_input")

blur_level = 5
if app_mode == "Background Blur":
    blur_level = st.slider("Select Background Blur Intensity Level", min_value=1, max_value=30, value=10, step=1)

if st.button("Process Image", type="primary"):
    if uploaded_file is None:
        st.warning("Please Upload a Valid Photo File to Edit")
    else:
        with st.spinner(f"Processing your Image ..."):
            try:
                input_image = Image.open(uploaded_file).convert("RGBA")
                
                if app_mode == "Background Removal":
                    processed_img = remove(input_image)
                    st.session_state['processed_photo_output'] = processed_img
                    
                elif app_mode == "Background Blur":
                    fg_mask = remove(input_image)
                    
                    bg_blurred = input_image.convert("RGB").filter(ImageFilter.GaussianBlur(radius=blur_level))
                    bg_blurred = bg_blurred.convert("RGBA")
                    
                    final_composite = Image.alpha_composite(bg_blurred, fg_mask)
                    st.session_state['processed_photo_output'] = final_composite

            except Exception as e:
                st.error(f"Failed to Process your Image")

if 'processed_photo_output' in st.session_state:
    st.markdown("---")
    st.image(st.session_state['processed_photo_output'], caption=f"Processed Output: {app_mode}", use_container_width=True)
    st.markdown("---")
    
    custom_name = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_editor")
    
    base_name = custom_name.strip() if custom_name.strip() else f"processed_{app_mode.lower().replace(' ', '_')}"
    clean_filename = f"{base_name}.png"
    
    img_byte_arr = io.BytesIO()
    st.session_state['processed_photo_output'].save(img_byte_arr, format='PNG')
    byte_payload = img_byte_arr.getvalue()
    
    st.download_button(
        label="Export your Edited Photo",
        data=byte_payload,
        file_name=clean_filename,
        mime="image/png",
        use_container_width=True,
        key="download_btn_editor",
        on_click="ignore"
    )